<a href="https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/exercicios/aula2-exercicio3-pular-quadros.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# Aula 2, exercício 3: Pular quadros

Processar todo quadro é caro. Nos primeiros 50 quadros, rode o modelo só em 1 a cada 5 (quadros 0, 5, 10...) e some quantas cadeiras (`chair`) foram detectadas.

O exercício é da aula [2. Detecção em vídeo](https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/notebooks/02-deteccao-em-video.ipynb), e este notebook roda sozinho. Use **Ambiente de execução → Executar tudo**: ele prepara os dados, deixa uma célula para você tentar e, no fim, mostra uma solução já conferida.

In [1]:
!pip install -q ultralytics

In [2]:
import base64
import subprocess
import urllib.request
from pathlib import Path

from IPython.display import HTML, display


def baixar(url, destino):
    if not Path(destino).exists():
        urllib.request.urlretrieve(url, destino)
    print(f"{destino}: {Path(destino).stat().st_size / 1e6:.1f} MB")
    return destino


def mostrar_video(caminho, largura=720):
    """Converte para H.264 (o navegador não toca mp4v) e embute no notebook."""
    web = str(Path(caminho).with_suffix("")) + "_web.mp4"
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", caminho,
         "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", web],
        check=True,
    )
    dados = base64.b64encode(Path(web).read_bytes()).decode()
    display(HTML(f'<video width="{largura}" controls autoplay muted loop>'
                 f'<source src="data:video/mp4;base64,{dados}" type="video/mp4"></video>'))
    return web


def baixar_para_maquina(caminho):
    try:
        from google.colab import files
        files.download(caminho)
    except ImportError:
        print(f"Fora do Colab: o arquivo está em {Path(caminho).resolve()}")

## Preparação

A preparação baixa o vídeo da sala e roda a detecção nos primeiros 150 quadros, como na aula 2. No fim ficam prontos o `contagem`, com as detecções por classe, e o `quadros`, com quantos foram processados.

In [3]:
from collections import Counter

import cv2
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
video = baixar(
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/home-objects-compressed.mp4",
    "sala.mp4",
)

MAX_QUADROS = 150
cap = cv2.VideoCapture(video)
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 25
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
contagem = Counter()
quadros = 0
while quadros < MAX_QUADROS:
    ok, frame = cap.read()
    if not ok:
        break
    r = model(frame, verbose=False)[0]
    contagem.update(model.names[int(c)] for c in r.boxes.cls)
    quadros += 1
cap.release()

print(f"{largura}x{altura} a {fps} FPS, {quadros} quadros processados")
print(dict(contagem.most_common(5)))

sala.mp4: 0.7 MB


1280x720 a 30 FPS, 140 quadros processados
{'chair': 311, 'potted plant': 259, 'dining table': 121, 'vase': 79, 'book': 65}


In [4]:
# @title Correção automática (rode, não precisa ler)
from functools import lru_cache

_cap = cv2.VideoCapture(video)
_total = int(_cap.get(cv2.CAP_PROP_FRAME_COUNT))
_fps = _cap.get(cv2.CAP_PROP_FPS)
_cap.release()


@lru_cache(None)
def _classes_por_quadro(n):
    cap = cv2.VideoCapture(video)
    saida = []
    for _ in range(n):
        ok, frame = cap.read()
        if not ok:
            break
        saida.append(tuple(model.names[int(c)] for c in model(frame, verbose=False)[0].boxes.cls))
    cap.release()
    return saida


def _quadros_do_arquivo(caminho):
    cap = cv2.VideoCapture(str(caminho))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return n


VERIFICAR = {
    1: lambda r: abs(float(r) - _total / _fps) < 0.05,
    2: lambda r: r == contagem.most_common(1)[0][0],
    3: lambda r: int(r) == sum(q.count("chair") for q in _classes_por_quadro(60)[:50:5]),
    4: lambda r: int(r) == sum("chair" in q for q in _classes_por_quadro(60)),
    5: lambda r: _quadros_do_arquivo(r) == 60,
    6: lambda r: abs(float(r) - contagem["chair"] / quadros) < 1e-3,
}

_acertos = {}


def conferir(n, resposta):
    if resposta is None:
        print(f"Exercício {n}: ainda sem resposta.")
        return
    try:
        certo = bool(VERIFICAR[n](resposta))
    except Exception as erro:
        certo = False
        print(f"  (a verificação deu erro: {erro})")
    _acertos[n] = certo
    print(f"Exercício {n}: " + ("✅ certo!" if certo else "❌ ainda não. Releia o enunciado e tente de novo."))


def placar():
    print(f"{sum(_acertos.values())} de {len(VERIFICAR)} exercícios certos.")

## Sua vez

Processar todo quadro é caro. Nos primeiros 50 quadros, rode o modelo só em 1 a cada 5 (quadros 0, 5, 10...) e some quantas cadeiras (`chair`) foram detectadas.

*Dica:* Leia todos os quadros com cap.read(), mas só chame o modelo quando i % 5 == 0.

Troque o `None` pela sua resposta e rode a célula.

In [5]:
resposta = None

conferir(3, resposta)

Exercício 3: ainda sem resposta.


## Uma solução

Compare com o que você fez. Existem outros caminhos certos: o que vale é a `conferir` aprovar.

In [6]:
cap = cv2.VideoCapture(video)
resposta = 0
for i in range(50):
    ok, frame = cap.read()
    if not ok:
        break
    if i % 5 == 0:
        r = model(frame, verbose=False)[0]
        resposta += sum(model.names[int(c)] == "chair" for c in r.boxes.cls)
cap.release()

conferir(3, resposta)

Exercício 3: ✅ certo!


## O que fica

Próximo: [Aula 2, exercício 4: Quadros com cadeira](https://colab.research.google.com/github/naubergois/curso-yolo-colab/blob/main/exercicios/aula2-exercicio4-quadros-com-cadeira.ipynb).